# Test results by image quality

The same test-split inference as `inference_visualization.ipynb`, but every number
is reported **separately for Good, Medium and Poor images** rather than pooled.

CAMUS ships an expert quality grade per patient-view in `Info_{view}.cfg`. A single
headline Dice averages across all three tiers, which is the one number a reader
cannot act on: the tier that decides whether a model is deployable is the worst
one, and it is also the smallest, so pooling hides it twice over — once by
averaging it away, and once by making it look like noise.

Each section here gives the full per-class metric table per tier, the gap between
tiers with uncertainty attached, and the cases behind it.

In [ ]:
%load_ext autoreload
%autoreload 2

import json
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.lines import Line2D
from torch.utils.data import DataLoader

from camus_dataset import LABELS, NUM_CLASSES, CAMUSDataset
from evaluation import METRIC_NAMES, SegmentationEvaluator
from model import build_unet, count_parameters, resolve_device

DATA_ROOT = Path.cwd().parent / "CAMUS_public"
RUN_DIR = Path("runs/unet_baseline/test2")
CHECKPOINT = RUN_DIR / "best.pt"
SPLIT = "test"

#: Worst last - every table and figure reads left to right as quality degrades.
QUALITIES = ("Good", "Medium", "Poor")
QUALITY_COLORS = {"Good": "#2fb8a0", "Medium": "#e8a24a", "Poor": "#e8564a"}

assert DATA_ROOT.is_dir(), f"CAMUS_public not found at {DATA_ROOT} - fix DATA_ROOT"
assert CHECKPOINT.is_file(), f"no checkpoint at {CHECKPOINT} - fix RUN_DIR"

plt.rcParams["figure.dpi"] = 110
DEVICE = resolve_device()
print(f"checkpoint  {CHECKPOINT}")
print(f"device      {DEVICE}")

## 1. Model, data, and one inference pass

Predictions are cached alongside the metrics so the qualitative sections later can
show the cases behind a number without a second forward pass.

In [ ]:
checkpoint = torch.load(CHECKPOINT, map_location=DEVICE, weights_only=True)
cfg = checkpoint["config"]

model = build_unet().to(DEVICE)
model.load_state_dict(checkpoint["model"])
model.eval()

test_ds = CAMUSDataset(DATA_ROOT, split=SPLIT, image_size=(cfg["image_size"],) * 2)
test_loader = DataLoader(test_ds, batch_size=cfg["batch_size"], shuffle=False, num_workers=4)

evaluator = SegmentationEvaluator(compute_distances=True)
cases = {}

with torch.inference_mode():
    for batch in test_loader:
        logits = model(batch["image"].to(DEVICE))
        evaluator.update(
            logits=logits,
            labels=batch["label"],
            spacing=batch["spacing"],
            meta={key: batch[key]
                  for key in ("key", "patient", "view", "instant", "image_quality")
                  if key in batch},
        )
        preds = logits.argmax(1).cpu().numpy()
        for i, key in enumerate(batch["key"]):
            cases[key] = {
                "image": batch["image"][i, 0].numpy(),
                "label": batch["label"][i].numpy(),
                "pred": preds[i],
                "quality": batch["image_quality"][i],
                "view": batch["view"][i],
                "instant": batch["instant"][i],
            }

print(f"epoch {checkpoint['epoch']}  ·  val_dice {checkpoint['val_dice']:.4f}  ·  "
      f"{count_parameters(model):,} parameters")
print(f"{len(cases)} test images scored")

## 2. What the split actually contains

Before comparing tiers, look at how they are composed. Quality is graded per
patient-view, so a patient can be Good in 4CH and Poor in 2CH — and if the Poor
tier is mostly 2CH images, part of any "Poor is worse" effect is really the 2CH
view being harder. Section 5 separates the two; this table is what makes the
question visible.

In [ ]:
def crosstab(field):
    counts = Counter((case["quality"], case[field]) for case in cases.values())
    values = sorted({case[field] for case in cases.values()})
    width = 10
    print(f"{'quality':<{width}}" + "".join(f"{v:>8}" for v in values) + f"{'total':>8}{'share':>8}")
    print("-" * (width + 8 * len(values) + 16))
    for quality in QUALITIES:
        row = [counts[(quality, v)] for v in values]
        print(f"{quality:<{width}}" + "".join(f"{c:>8}" for c in row)
              + f"{sum(row):>8}{sum(row) / len(cases):>7.0%}")
    print()


crosstab("view")
crosstab("instant")

patient_quality = {}
for key, case in cases.items():
    patient_quality.setdefault(key.split("_")[0], set()).add(case["quality"])
mixed = {p: sorted(q) for p, q in patient_quality.items() if len(q) > 1}
print(f"patients whose two views are graded differently: {len(mixed)}/{len(patient_quality)}")
print(f"  e.g. {dict(list(mixed.items())[:4])}")

## 3. Full metric report per tier

`SegmentationEvaluator` aggregates macro over cases, so a tier's numbers are the
mean over its own images and nothing else. Splitting the accumulated per-case
results is exactly what `evaluator.stratified()` does internally; here the child
evaluators are kept so each tier can print the complete per-class table.

In [ ]:
def evaluator_for(results):
    """A child evaluator over a subset of the accumulated cases."""
    child = SegmentationEvaluator(
        num_classes=evaluator.num_classes,
        include_background=evaluator.include_background,
        compute_distances=evaluator.compute_distances,
        hd_percentile=evaluator.hd_percentile,
    )
    child.results = results
    return child


groups = {
    quality: evaluator_for([c for c in evaluator.results if c.meta["image_quality"] == quality])
    for quality in QUALITIES
}
summaries = {quality: group.compute() for quality, group in groups.items()}

print("=" * 74)
print(f"ALL IMAGES  (n = {len(evaluator.results)})")
print("=" * 74)
print(evaluator.report())

for quality in QUALITIES:
    print("\n" + "=" * 74)
    print(f"{quality.upper()}  (n = {summaries[quality]['n_cases']}, "
          f"{summaries[quality]['n_cases'] / len(evaluator.results):.0%} of the split)")
    print("=" * 74)
    print(groups[quality].report(summaries[quality]))

## 4. Side by side, with uncertainty

The Poor tier is the smallest, so the honest comparison carries an interval. These
are bootstrap percentile CIs over cases (resampling the tier's own images), which
make no assumption about the metric being normally distributed — Dice is bounded
and left-skewed, so a mean ± sd interval would run off the top of the scale.

An overlap between two tiers' intervals means the split has not shown a difference,
not that there isn't one.

In [ ]:
def per_case(group, metric, class_name=None):
    """Per-case values of `metric`, class-averaged unless a class is named."""
    names = [class_name] if class_name else list(evaluator.class_names)
    values = [np.nanmean([case.metrics[n][metric] for n in names]) for case in group.results]
    return np.array(values, dtype=float)


def bootstrap_ci(values, n_resamples=4000, seed=0):
    finite = np.asarray(values)[np.isfinite(values)]
    if finite.size == 0:
        return float("nan"), float("nan")
    draws = np.random.default_rng(seed).choice(finite, size=(n_resamples, finite.size))
    return tuple(float(v) for v in np.percentile(draws.mean(axis=1), (2.5, 97.5)))


for metric, fmt in (("dice", "{:.4f}"), ("iou", "{:.4f}"), ("hd95", "{:.2f}"), ("assd", "{:.2f}")):
    print(f"\n{metric.upper()}   mean [95% CI]")
    print(f"{'class':<18}" + "".join(f"{q:>26}" for q in QUALITIES) + f"{'Good - Poor':>14}")
    print("-" * (18 + 26 * len(QUALITIES) + 14))
    for class_name in (*evaluator.class_names, "mean"):
        cells, means = [], {}
        for quality in QUALITIES:
            values = per_case(groups[quality], metric, None if class_name == "mean" else class_name)
            mean = float(np.nanmean(values))
            low, high = bootstrap_ci(values)
            means[quality] = mean
            cells.append(f"{fmt.format(mean)} [{fmt.format(low)}, {fmt.format(high)}]".rjust(26))
        gap = means["Good"] - means["Poor"]
        print(f"{class_name:<18}" + "".join(cells) + f"{gap:>+14.4f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), layout="constrained")
bar_width = 0.26
positions = np.arange(len(evaluator.class_names))

for ax, metric, label in (
    (axes[0], "dice", "Dice"),
    (axes[1], "hd95", "HD95 (mm)"),
    (axes[2], "assd", "ASSD (mm)"),
):
    for offset, quality in zip((-bar_width, 0, bar_width), QUALITIES):
        means, errors = [], [[], []]
        for class_name in evaluator.class_names:
            values = per_case(groups[quality], metric, class_name)
            mean = float(np.nanmean(values))
            low, high = bootstrap_ci(values)
            means.append(mean)
            errors[0].append(mean - low); errors[1].append(high - mean)
        ax.bar(positions + offset, means, bar_width, yerr=errors, capsize=3,
               color=QUALITY_COLORS[quality], alpha=0.85, label=quality,
               error_kw={"lw": 1, "ecolor": "0.3"})
    ax.set_xticks(positions, [n.replace("_", "\n") for n in evaluator.class_names], fontsize=8)
    ax.set_ylabel(label)
    ax.set_title(f"{label} by image quality", fontsize=10)
    ax.grid(axis="y", alpha=0.25)
axes[0].set_ylim(0, 1.0)
axes[0].legend(fontsize=8, loc="lower right")
fig.suptitle(f"Test split by expert image-quality grade · {CHECKPOINT}", fontsize=11)
plt.show()

## 5. Is it quality, or is it the view?

2CH is the harder view in this split regardless of grade, so the tier comparison is
only meaningful if it survives holding the view fixed. If the Good-to-Poor gap
persists inside both views, quality is carrying real signal; if it collapses, the
tiers were largely a proxy for which view they contain.

In [ ]:
print(f"{'view':<8}{'quality':<10}{'n':>5}{'Dice':>9}{'HD95':>9}{'ASSD':>9}")
print("-" * 50)
within = {}
for view in ("2CH", "4CH"):
    for quality in QUALITIES:
        subset = [c for c in evaluator.results
                  if c.meta["view"] == view and c.meta["image_quality"] == quality]
        if not subset:
            continue
        summary = evaluator_for(subset).compute()
        within[(view, quality)] = summary
        print(f"{view:<8}{quality:<10}{summary['n_cases']:>5}{summary['mean_dice']:>9.4f}"
              f"{summary['mean_hd95']:>9.3f}{summary['mean_assd']:>9.3f}")
    print()

for view in ("2CH", "4CH"):
    gap = within[(view, "Good")]["mean_dice"] - within[(view, "Poor")]["mean_dice"]
    print(f"{view}: Good - Poor = {gap:+.4f} Dice")

## 6. Distributions

Means say where a tier sits; the spread says how often it fails. A tier whose
median is fine but whose lower tail reaches 0.6 is a tier that produces occasional
unusable contours, and that is a different clinical risk from one that is uniformly
slightly worse.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), layout="constrained")

mean_dice = {q: per_case(groups[q], "dice") for q in QUALITIES}
parts = axes[0].violinplot([mean_dice[q][np.isfinite(mean_dice[q])] for q in QUALITIES],
                           showmedians=True, widths=0.8)
for body, quality in zip(parts["bodies"], QUALITIES):
    body.set_facecolor(QUALITY_COLORS[quality]); body.set_alpha(0.6)
for x, quality in enumerate(QUALITIES, start=1):
    values = mean_dice[quality]
    axes[0].scatter(np.random.default_rng(0).normal(x, 0.045, values.size), values,
                    s=9, alpha=0.5, color="0.25", zorder=3)
axes[0].set_xticks(range(1, len(QUALITIES) + 1), QUALITIES)
# Every case sits above 0.65, so a 0-1 axis would spend four fifths of the
# figure on empty space and squash the gap the plot exists to show.
floor = float(np.floor(min(np.nanmin(v) for v in mean_dice.values()) * 20) / 20)
axes[0].set_ylabel("mean Dice"); axes[0].set_ylim(floor, 1.005)
axes[0].set_title("per-case Dice by quality", fontsize=10)
axes[0].grid(axis="y", alpha=0.25)

# Cumulative view: for any threshold, what share of each tier clears it?
for quality in QUALITIES:
    values = np.sort(mean_dice[quality][np.isfinite(mean_dice[quality])])
    axes[1].step(values, 1 - np.arange(values.size) / values.size, where="post",
                 color=QUALITY_COLORS[quality], lw=1.8, label=quality)
axes[1].set_xlabel("Dice threshold"); axes[1].set_ylabel("share of tier at or above")
axes[1].set_xlim(floor, 1.0); axes[1].set_ylim(0, 1.02)
axes[1].set_title("survival curve: how often a tier clears a bar", fontsize=10)
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.25)

hd = {q: per_case(groups[q], "hd95") for q in QUALITIES}
axes[2].boxplot([hd[q][np.isfinite(hd[q])] for q in QUALITIES],
                tick_labels=list(QUALITIES), showfliers=True,
                flierprops={"marker": ".", "markersize": 4})
axes[2].set_ylabel("HD95 (mm)")
axes[2].set_title("boundary error by quality", fontsize=10)
axes[2].grid(axis="y", alpha=0.25)
plt.show()

print("share of cases below 0.85 mean Dice:")
for quality in QUALITIES:
    values = mean_dice[quality]
    print(f"  {quality:<7} {np.mean(values < 0.85):6.1%}  ({int((values < 0.85).sum())}/{values.size})")

## 7. The cases behind the numbers

Worst three per tier. A Poor-tier failure that looks like a shadowed, genuinely
ambiguous image is a data limit; the same failure on a Good-tier image is a model
limit, and only looking at them tells the two apart.

In [ ]:
CLASS_COLORS = {1: "#e8564a", 2: "#2fb8a0", 3: "#4a7fe8"}


def contours(ax, label, linestyle):
    # Outermost first so the endocardial border stays on top.
    for cls in sorted(CLASS_COLORS, reverse=True):
        mask = label == cls
        if mask.any():
            ax.contour(mask.astype(float), levels=[0.5], colors=[CLASS_COLORS[cls]],
                       linewidths=1.1, linestyles=linestyle)


def show_row(axes, keys, row_label):
    for ax, key in zip(axes, keys):
        case = cases[key]
        ax.imshow(case["image"], cmap="gray")
        contours(ax, case["label"], "-")
        contours(ax, case["pred"], "--")
        scores = [c for c in evaluator.results if c.key == key][0].metrics
        dice = np.nanmean([scores[n]["dice"] for n in evaluator.class_names])
        ax.set_title(f"{key}\n{case['view']} {case['instant']}  Dice {dice:.3f}", fontsize=7.5)
    for ax in axes:
        ax.set_xticks([]); ax.set_yticks([])
    axes[0].set_ylabel(row_label, fontsize=10, color=QUALITY_COLORS[row_label])


fig, axes = plt.subplots(len(QUALITIES), 3, figsize=(9.6, 3.6 * len(QUALITIES)),
                         layout="constrained", squeeze=False)
for row, quality in zip(axes, QUALITIES):
    worst = groups[quality].worst_cases(n=3, metric="dice")
    show_row(row, [key for key, _ in worst], quality)

fig.suptitle("worst three cases in each quality tier", fontsize=11)
fig.legend(handles=[Line2D([0], [0], color=CLASS_COLORS[c], lw=3, label=f"{c} {LABELS[c]}")
                    for c in CLASS_COLORS]
                   + [Line2D([0], [0], color="0.35", lw=1.5, ls=ls, label=tag)
                      for ls, tag in (("-", "expert"), ("--", "model"))],
           loc="outside lower center", ncol=5, fontsize=8, frameon=False)
plt.show()

In [ ]:
# The median case of each tier, for what "typical" looks like at that grade.
fig, axes = plt.subplots(1, len(QUALITIES), figsize=(3.4 * len(QUALITIES), 4.0),
                         layout="constrained")
for ax, quality in zip(axes, QUALITIES):
    scored = sorted(((c.key, np.nanmean([c.metrics[n]["dice"] for n in evaluator.class_names]))
                     for c in groups[quality].results), key=lambda kv: kv[1])
    key, dice = scored[len(scored) // 2]
    case = cases[key]
    ax.imshow(case["image"], cmap="gray")
    contours(ax, case["label"], "-")
    contours(ax, case["pred"], "--")
    ax.set_title(f"{quality} · median case\n{key}  Dice {dice:.3f}", fontsize=8.5,
                 color=QUALITY_COLORS[quality])
    ax.set_xticks([]); ax.set_yticks([])
plt.show()

## 8. Save

Written next to the checkpoint, one entry per tier plus the within-view breakdown
that shows whether the tiers are carrying view composition.

In [ ]:
report_path = RUN_DIR / f"{SPLIT}_metrics_by_quality.json"
report_path.write_text(json.dumps({
    "checkpoint": str(CHECKPOINT),
    "epoch": checkpoint["epoch"],
    "split": SPLIT,
    "overall": evaluator.compute(),
    "by_quality": summaries,
    "by_quality_ci": {
        quality: {
            metric: dict(zip(("low", "high"), bootstrap_ci(per_case(groups[quality], metric))))
            for metric in METRIC_NAMES
        }
        for quality in QUALITIES
    },
    "by_view_and_quality": {f"{view}/{quality}": summary
                            for (view, quality), summary in within.items()},
    "composition": {quality: summaries[quality]["n_cases"] for quality in QUALITIES},
}, indent=2, default=str))
print(f"wrote {report_path}")